In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random, math
import os # Import the os module

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Create the output directory if it doesn't exist
OUTPUT_DIR = "/mnt/user-data/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# ██████████████████████████████████████████████████████████████████████████
# PART 1 — ENGLISH TO URDU TRANSLATION
# ██████████████████████████████████████████████████████████████████████████
# =============================================================================

print("\n" + "═"*65)
print("  PART 1 — English → Urdu Translation (Encoder-Decoder RNN)")
print("═"*65)

# =============================================================================
# 1-A  CUSTOM DATASET  (Task 1)
# =============================================================================

english_sentences = [
    "hello how are you",
    "i am fine thank you",
    "what is your name",
    "i love programming",
    "good morning",
    "good night",
    "where are you going",
    "i am going to school",
    "what time is it",
    "i am hungry",
    "please help me",
    "thank you very much",
    "i like to read books",
    "the weather is nice today",
    "i want to learn urdu",
    "pakistan is a beautiful country",
    "where do you live",
    "i live in karachi",
    "do you speak english",
    "yes i speak english",
]

urdu_sentences = [
    "<sos> ہیلو آپ کیسے ہیں <eos>",
    "<sos> میں ٹھیک ہوں شکریہ <eos>",
    "<sos> آپ کا نام کیا ہے <eos>",
    "<sos> مجھے پروگرامنگ پسند ہے <eos>",
    "<sos> صبح بخیر <eos>",
    "<sos> شب بخیر <eos>",
    "<sos> آپ کہاں جا رہے ہیں <eos>",
    "<sos> میں اسکول جا رہا ہوں <eos>",
    "<sos> کیا وقت ہوا ہے <eos>",
    "<sos> مجھے بھوک لگی ہے <eos>",
    "<sos> براہ کرم میری مدد کریں <eos>",
    "<sos> بہت بہت شکریہ <eos>",
    "<sos> مجھے کتابیں پڑھنا پسند ہے <eos>",
    "<sos> آج موسم اچھا ہے <eos>",
    "<sos> میں اردو سیکھنا چاہتا ہوں <eos>",
    "<sos> پاکستان ایک خوبصورت ملک ہے <eos>",
    "<sos> آپ کہاں رہتے ہیں <eos>",
    "<sos> میں کراچی میں رہتا ہوں <eos>",
    "<sos> کیا آپ انگریزی بولتے ہیں <eos>",
    "<sos> جی ہاں میں انگریزی بولتا ہوں <eos>",
]

# =============================================================================
# 1-B  VOCABULARY BUILDER
# =============================================================================

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

def build_vocab(sentences):
    vocab = {PAD: 0, SOS: 1, EOS: 2, UNK: 3}
    for sent in sentences:
        for tok in sent.split():
            if tok not in vocab and tok not in (SOS, EOS):
                vocab[tok] = len(vocab)
    return vocab

en_vocab  = build_vocab(english_sentences)
ur_vocab  = build_vocab(urdu_sentences)
en_i2w    = {v: k for k, v in en_vocab.items()}
ur_i2w    = {v: k for k, v in ur_vocab.items()}

EN_VOCAB  = len(en_vocab)
UR_VOCAB  = len(ur_vocab)
MAX_LEN   = max(max(len(s.split()) for s in english_sentences),
                max(len(s.split()) for s in urdu_sentences)) + 2

def encode(sent, vocab, add_sos=False, add_eos=False):
    tokens = []
    if add_sos:
        tokens.append(vocab[SOS])
    tokens += [vocab.get(t, vocab[UNK]) for t in sent.split()
               if t not in (SOS, EOS)]
    if add_eos:
        tokens.append(vocab[EOS])
    return tokens

def pad_seq(seq, max_len):
    return seq + [en_vocab[PAD]] * (max_len - len(seq))

X_en = [pad_seq(encode(s, en_vocab, add_sos=False, add_eos=False), MAX_LEN)
        for s in english_sentences]
# decoder input  = <sos> + urdu words   (teacher forcing)
# decoder target = urdu words + <eos>
y_in  = [pad_seq(encode(s, ur_vocab, add_sos=True,  add_eos=False), MAX_LEN)
         for s in urdu_sentences]
y_out = [pad_seq(encode(s, ur_vocab, add_sos=False, add_eos=True),  MAX_LEN)
         for s in urdu_sentences]

X_t   = torch.tensor(X_en,  dtype=torch.long).to(DEVICE)
Yi_t  = torch.tensor(y_in,  dtype=torch.long).to(DEVICE)
Yo_t  = torch.tensor(y_out, dtype=torch.long).to(DEVICE)

print(f"  English vocab  : {EN_VOCAB}  |  Urdu vocab : {UR_VOCAB}")
print(f"  Sentence pairs : {len(english_sentences)}  |  MAX_LEN : {MAX_LEN}")

# =============================================================================
# 1-C  DATASET & DATALOADER
# =============================================================================

class TranslationDataset(Dataset):
    def __init__(self, X, Yi, Yo):
        self.X, self.Yi, self.Yo = X, Yi, Yo
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Yi[i], self.Yo[i]

tr_dataset = TranslationDataset(X_t, Yi_t, Yo_t)
tr_loader  = DataLoader(tr_dataset, batch_size=4, shuffle=True)

# =============================================================================
# 1-D  ENCODER – DECODER MODEL
# =============================================================================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_units, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn       = nn.GRU(embed_dim, hidden_units, batch_first=True) # Changed from nn.RNN to nn.GRU
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))   # (B, T, E)
        _, hidden = self.rnn(emb)                # hidden: (1, B, H)
        return hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_units, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn       = nn.GRU(embed_dim, hidden_units, batch_first=True) # Changed from nn.RNN to nn.GRU
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_units, vocab_size)

    def forward(self, x, hidden):
        emb = self.dropout(self.embedding(x))    # (B, T, E)
        out, hidden = self.rnn(emb, hidden)      # (B, T, H)
        logits = self.fc(out)                    # (B, T, V)
        return logits, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden = self.encoder(src)
        logits, _ = self.decoder(trg, hidden)
        return logits

# =============================================================================
# 1-E  TRAINING FUNCTION  (Task 2 — vary units / epochs / lr)
# =============================================================================

def train_translation(units, epochs, lr, label):
    embed_dim = 64
    enc = Encoder(EN_VOCAB, embed_dim, units).to(DEVICE)
    dec = Decoder(UR_VOCAB, embed_dim, units).to(DEVICE)
    model = Seq2Seq(enc, dec).to(DEVICE)

    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    loss_hist, acc_hist = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for src, trg_in, trg_out in tr_loader:
            optimizer.zero_grad()
            logits = model(src, trg_in)               # (B, T, V)
            loss   = criterion(
                logits.view(-1, UR_VOCAB), trg_out.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

            mask    = trg_out != 0
            preds   = logits.argmax(-1)
            correct += (preds[mask] == trg_out[mask]).sum().item()
            total   += mask.sum().item()

        loss_hist.append(total_loss / len(tr_loader))
        acc_hist.append(correct / total if total > 0 else 0)

        if epoch % max(1, epochs // 5) == 0 or epoch == epochs:
            print(f"  [{label}] Ep {epoch:>3}/{epochs} | "
                  f"Loss {loss_hist[-1]:.4f} | Acc {acc_hist[-1]:.4f}")

    print()
    return model, loss_hist, acc_hist

# =============================================================================
# 1-F  RUN EXPERIMENTS  (Task 2)
# =============================================================================

configs_trans = [
    {"units": 64,  "epochs": 30, "lr": 0.01,   "label": "Small  (64u,  lr=1e-2)"},
    {"units": 128, "epochs": 50, "lr": 0.001,  "label": "Medium (128u, lr=1e-3)"},
    {"units": 256, "epochs": 80, "lr": 0.0005, "label": "Large  (256u, lr=5e-4)"},
]

trans_results = {}
for cfg in configs_trans:
    print("─"*65)
    print(f"  Config: {cfg['label']}")
    print("─"*65)
    model_t, l_h, a_h = train_translation(
        cfg["units"], cfg["epochs"], cfg["lr"], cfg["label"])
    trans_results[cfg["label"]] = {
        "model": model_t, "loss": l_h, "acc": a_h, "epochs": cfg["epochs"]}

# =============================================================================
# 1-G  TRANSLATION FUNCTION
# =============================================================================

def translate(model, sentence, max_gen=MAX_LEN):
    model.eval()
    tokens = [en_vocab.get(t, en_vocab[UNK]) for t in sentence.lower().split()]
    tokens = tokens[:MAX_LEN]
    padded = tokens + [en_vocab[PAD]] * (MAX_LEN - len(tokens))
    src    = torch.tensor([padded], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        hidden = model.encoder(src)
        dec_in = torch.tensor([[ur_vocab[SOS]]], dtype=torch.long).to(DEVICE)
        generated = []
        for _ in range(max_gen):
            logits, hidden = model.decoder(dec_in, hidden)
            pred = logits.argmax(-1).item()
            if pred == ur_vocab[EOS] or pred == ur_vocab[PAD]:
                break
            word = ur_i2w.get(pred, UNK)
            if word not in (SOS, EOS, PAD):
                generated.append(word)
            dec_in = torch.tensor([[pred]], dtype=torch.long).to(DEVICE)

    return " ".join(generated)

# =============================================================================
# 1-H  TEST TRANSLATIONS
# =============================================================================

best_trans_model = trans_results["Large  (256u, lr=5e-4)"]["model"]

test_en = [
    "hello how are you",
    "i love programming",
    "good morning",
    "i am hungry",
    "thank you very much",
    "pakistan is a beautiful country",
]

print("═"*65)
print("  TRANSLATION RESULTS  (best model)")
print("═"*65)
print(f"  {'English':<35} {'Urdu (Predicted)'}")
print("  " + "─"*60)
for sent in test_en:
    translation = translate(best_trans_model, sent)
    print(f"  {sent:<35} {translation}")

# =============================================================================
# 1-I  PLOT — Training Curves for Translation
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#e74c3c", "#2ecc71", "#3498db"]

for (lbl, res), c in zip(trans_results.items(), colors):
    ep = list(range(1, res["epochs"] + 1))
    axes[0].plot(ep, res["acc"],  label=lbl, color=c, lw=2)
    axes[1].plot(ep, res["loss"], label=lbl, color=c, lw=2, ls="--")

for ax, title, ylabel in zip(
        axes,
        ["Training Accuracy — Translation", "Training Loss — Translation"],
        ["Accuracy", "Loss"]):
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "translation_training_curves.png"), dpi=150)
plt.close()
print("\nSaved ▶  translation_training_curves.png")

# =============================================================================
# 1-J  PLOT — Encoder-Decoder Architecture Diagram
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14); ax.set_ylim(0, 7)
ax.axis('off'); ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('#f0f4f8')
ax.set_title("Encoder–Decoder RNN Architecture (English → Urdu)",
             fontsize=14, fontweight='bold', pad=15)

def draw_box(ax, x, y, w, h, color, text, fontsize=9):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, color='white', fontweight='bold')

# Encoder blocks
enc_xs = [1.2, 2.7, 4.2, 5.7]
enc_labels = ["Input\nEmbedding", "RNN\nCell t=1", "RNN\nCell t=2", "RNN\nCell t=n"]
for x, lbl in zip(enc_xs, enc_labels):
    draw_box(ax, x, 4.5, 1.2, 1.0, "#2980b9", lbl)

# Context vector
draw_box(ax, 7.2, 4.5, 1.4, 1.0, "#8e44ad", "Context\nVector\n(hidden)", 8)

# Decoder blocks
dec_xs = [9.0, 10.5, 12.0, 13.5]
dec_labels = ["RNN\nCell t=1", "RNN\nCell t=2", "RNN\nCell t=3", "RNN\nCell t=m"]
for x, lbl in zip(dec_xs, dec_labels):
    draw_box(ax, x, 4.5, 1.2, 1.0, "#27ae60", lbl)

# Output labels below decoder
out_lbls = ["صبح", "بخیر", "آج", "...."]
for x, lbl in zip(dec_xs, out_lbls):
    draw_box(ax, x, 2.8, 1.0, 0.7, "#f39c12", lbl, fontsize=10)
    ax.annotate("", xy=(x, 3.15), xytext=(x, 4.0),
                arrowprops=dict(arrowstyle="->", color="#f39c12", lw=1.5))

# Input words above encoder
in_words = ["hello", "how", "are", "you"]
for x, w in zip(enc_xs, in_words):
    draw_box(ax, x, 6.2, 1.0, 0.6, "#7f8c8d", w, fontsize=9)
    ax.annotate("", xy=(x, 5.0), xytext=(x, 5.9),
                arrowprops=dict(arrowstyle="->", color="#7f8c8d", lw=1.5))

# Arrows between encoder cells
for i in range(len(enc_xs) - 1):
    ax.annotate("", xy=(enc_xs[i+1] - 0.65, 4.5), xytext=(enc_xs[i] + 0.65, 4.5),
                arrowprops=dict(arrowstyle="->", color="white", lw=2))

# Arrow encoder → context
ax.annotate("", xy=(6.45, 4.5), xytext=(6.35, 4.5),
            arrowprops=dict(arrowstyle="->", color="#8e44ad", lw=2.5))

# Arrow context → first decoder
ax.annotate("", xy=(8.35, 4.5), xytext=(7.95, 4.5),
            arrowprops=dict(arrowstyle="->", color="#8e44ad", lw=2.5))

# Arrows between decoder cells
for i in range(len(dec_xs) - 1):
    ax.annotate("", xy=(dec_xs[i+1] - 0.65, 4.5), xytext=(dec_xs[i] + 0.65, 4.5),
                arrowprops=dict(arrowstyle="->", color="white", lw=2))

# Section labels
ax.text(3.4, 1.8, "ENCODER", ha='center', fontsize=11,
        color='#2980b9', fontweight='bold')
ax.text(11.2, 1.8, "DECODER", ha='center', fontsize=11,
        color='#27ae60', fontweight='bold')
ax.text(7.2, 1.8, "Context", ha='center', fontsize=9,
        color='#8e44ad', fontweight='bold')

# <sos> token label
ax.text(9.0, 6.1, "<sos>", ha='center', fontsize=8,
        color='#27ae60', fontstyle='italic')
ax.annotate("", xy=(9.0, 5.0), xytext=(9.0, 5.85),
            arrowprops=dict(arrowstyle="->", color="#27ae60", lw=1.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "encoder_decoder_architecture.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved ▶  encoder_decoder_architecture.png")


# =============================================================================
# ██████████████████████████████████████████████████████████████████████████
# PART 2 — ONE-TO-MANY RNN: BABY NAME GENERATION  (Task 3)
# ██████████████████████████████████████████████████████████████████████████
# =============================================================================

print("\n" + "═"*65)
print("  PART 2 — One-to-Many RNN: Baby Name Generation")
print("═"*65)

# =============================================================================
# 2-A  BABY NAMES DATASET
# =============================================================================

baby_names = [
    # Boys
    "muhammad", "ali", "ahmed", "omar", "hassan", "hussain", "ibrahim",
    "yusuf", "adam", "noah", "lucas", "liam", "ethan", "james", "oliver",
    "william", "henry", "jacob", "mason", "logan", "aiden", "caleb",
    "ryan", "nathan", "aaron", "daniel", "samuel", "david", "joseph",
    "michael", "elijah", "gabriel", "sebastian", "julian", "leo",
    # Girls
    "fatima", "aisha", "zainab", "maryam", "sara", "layla", "noor",
    "sophia", "emma", "olivia", "amelia", "isabella", "mia", "charlotte",
    "harper", "evelyn", "abigail", "emily", "ella", "elizabeth", "camila",
    "luna", "aria", "chloe", "penelope", "grace", "zoe", "nora", "lily",
    "eleanor", "hannah", "aurora", "savannah", "brooklyn", "stella",
    "victoria", "claire", "skylar", "isla", "genesis", "naomi",
]

# character vocab
all_chars  = sorted(set("".join(baby_names)))
char2idx   = {c: i+2 for i, c in enumerate(all_chars)}   # 0=PAD, 1=EOS
char2idx["<PAD>"] = 0
char2idx["<EOS>"] = 1
idx2char   = {v: k for k, v in char2idx.items()}
CHAR_VOCAB  = len(char2idx)

def name_to_tensor(name):
    ids = [char2idx[c] for c in name] + [char2idx["<EOS>"]]
    return torch.tensor(ids, dtype=torch.long)

print(f"  Baby names     : {len(baby_names)}")
print(f"  Char vocab     : {CHAR_VOCAB}  chars: {''.join(all_chars)}")

# =============================================================================
# 2-B  ONE-TO-MANY RNN MODEL
# =============================================================================

class BabyNameRNN(nn.Module):
    """
    One-to-Many: receives a single start embedding (zeros or learned),
    then generates one character at a time for max_len steps.
    """
    def __init__(self, vocab_size, embed_dim, hidden_units, dropout=0.3):
        super().__init__()
        self.hidden_units = hidden_units
        self.embedding    = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn          = nn.RNN(embed_dim, hidden_units, batch_first=True)
        self.dropout      = nn.Dropout(dropout)
        self.fc           = nn.Linear(hidden_units, vocab_size)

    def forward(self, x, hidden=None):
        """x: (B, T)  — teacher-forced character sequence"""
        emb    = self.dropout(self.embedding(x))   # (B, T, E)
        out, h = self.rnn(emb, hidden)              # (B, T, H)
        logits = self.fc(self.dropout(out))         # (B, T, V)
        return logits, h

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_units).to(DEVICE)

# =============================================================================
# 2-C  TRAINING  (character-level language model)
# =============================================================================

class NamesDataset(Dataset):
    def __init__(self, names):
        self.data = [name_to_tensor(n) for n in names]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate_names(batch):
    max_l = max(b.size(0) for b in batch)
    X, Y  = [], []
    for t in batch:
        inp = torch.cat([torch.tensor([char2idx["<EOS>"]]), t[:-1]])  # shifted right
        inp = torch.nn.functional.pad(inp, (0, max_l - inp.size(0)))
        tgt = torch.nn.functional.pad(t,   (0, max_l - t.size(0)))
        X.append(inp); Y.append(tgt)
    return torch.stack(X), torch.stack(Y)

names_loader = DataLoader(
    NamesDataset(baby_names), batch_size=8,
    shuffle=True, collate_fn=collate_names)

def train_namegen(units, epochs, lr, label):
    model     = BabyNameRNN(CHAR_VOCAB, embed_dim=32, hidden_units=units).to(DEVICE)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    loss_hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for X_b, Y_b in names_loader:
            X_b, Y_b = X_b.to(DEVICE), Y_b.to(DEVICE)
            h = model.init_hidden(X_b.size(0))
            optimizer.zero_grad()
            logits, _ = model(X_b, h)
            loss = criterion(logits.reshape(-1, CHAR_VOCAB), Y_b.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        loss_hist.append(total_loss / len(names_loader))

        if epoch % max(1, epochs // 5) == 0 or epoch == epochs:
            print(f"  [{label}] Epoch {epoch:>3}/{epochs} | Loss {loss_hist[-1]:.4f}")

    print()
    return model, loss_hist

# =============================================================================
# 2-D  EXPERIMENTS — vary units / epochs / lr  (Task 2 for name gen)
# =============================================================================

name_configs = [
    {"units": 64,  "epochs": 40,  "lr": 0.01,   "label": "Small  (64u,  lr=1e-2)"},
    {"units": 128, "epochs": 80,  "lr": 0.005,  "label": "Medium (128u, lr=5e-3)"},
    {"units": 256, "epochs": 120, "lr": 0.001,  "label": "Large  (256u, lr=1e-3)"},
]

name_results = {}
for cfg in name_configs:
    print("─"*65)
    print(f"  Config: {cfg['label']}")
    print("─"*65)
    m, lh = train_namegen(cfg["units"], cfg["epochs"], cfg["lr"], cfg["label"])
    name_results[cfg["label"]] = {"model": m, "loss": lh, "epochs": cfg["epochs"]}

# =============================================================================
# 2-E  NAME GENERATION FUNCTION
# =============================================================================

def generate_name(model, seed_char=None, max_len=12, temperature=0.8):
    """
    One-to-Many: start from a single seed character (or random),
    generate characters until <EOS> or max_len.
    """
    model.eval()
    if seed_char is None:
        seed_char = random.choice(all_chars)

    generated = [seed_char]
    inp = torch.tensor([[char2idx[seed_char]]], dtype=torch.long).to(DEVICE)
    h   = model.init_hidden(1)

    with torch.no_grad():
        for _ in range(max_len - 1):
            logits, h = model(inp, h)                     # (1,1,V)
            logits    = logits[0, 0] / temperature         # scale
            probs     = torch.softmax(logits, dim=-1).cpu().numpy()
            next_idx  = np.random.choice(len(probs), p=probs)
            if next_idx == char2idx["<EOS>"]:
                break
            ch = idx2char.get(next_idx, "")
            if ch in ("<PAD>", "<EOS>"):
                break
            generated.append(ch)
            inp = torch.tensor([[next_idx]], dtype=torch.long).to(DEVICE)

    return "".join(generated).capitalize()

# =============================================================================
# 2-F  GENERATE NAMES
# =============================================================================

best_name_model = name_results["Large  (256u, lr=1e-3)"]["model"]

seeds  = list("abcdefghijklmnopqrstuvwxyz")
temps  = [0.5, 0.8, 1.0]

print("═"*65)
print("  GENERATED BABY NAMES  (best model)")
print("═"*65)

for temp in temps:
    names_gen = [generate_name(best_name_model, temperature=temp) for _ in range(10)]
    print(f"\n  Temperature = {temp}:")
    print("  " + "  |  ".join(names_gen))

# =============================================================================
# 2-G  PLOT — Name Gen Loss + Generated name length distribution
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors3 = ["#e74c3c", "#9b59b6", "#1abc9c"]

for (lbl, res), c in zip(name_results.items(), colors3):
    axes[0].plot(range(1, res["epochs"]+1), res["loss"],
                 label=lbl, color=c, lw=2)

axes[0].set_title("Name Generation — Training Loss", fontweight="bold", fontsize=12)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# name length distribution
gen_names = [generate_name(best_name_model, temperature=0.8) for _ in range(200)]
lengths   = [len(n) for n in gen_names]
axes[1].hist(lengths, bins=range(2, 16), color="#3498db", edgecolor='white',
             alpha=0.85, align='left')
axes[1].set_title("Distribution of Generated Name Lengths", fontweight="bold", fontsize=12)
axes[1].set_xlabel("Name Length (chars)"); axes[1].set_ylabel("Count")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "namegen_analysis.png"), dpi=150)
plt.close()
print("\nSaved ▶  namegen_analysis.png")

# =============================================================================
# 2-H  PLOT — One-to-Many RNN Architecture Diagram
# =============================================================================

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 6)
ax.axis('off'); ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('#f0f4f8')
ax.set_title("One-to-Many RNN — Baby Name Generation",
             fontsize=14, fontweight='bold', pad=12)

# single input
draw_box(ax, 1.0, 3.0, 1.3, 0.9, "#7f8c8d", "Seed\nChar 'A'", 9)

# RNN cells
cell_xs   = [3.0, 5.0, 7.0, 9.0, 11.0]
out_chars  = ["a", "l", "i", "a", "<EOS>"]
cell_color = "#8e44ad"

for i, (x, ch) in enumerate(zip(cell_xs, out_chars)):
    draw_box(ax, x, 3.0, 1.3, 0.9, cell_color, f"RNN\nt={i+1}", 9)
    draw_box(ax, x, 1.3, 0.9, 0.7, "#f39c12", ch, 10)
    ax.annotate("", xy=(x, 1.68), xytext=(x, 2.55),
                arrowprops=dict(arrowstyle="->", color="#f39c12", lw=1.8))

# arrow: input → first cell
ax.annotate("", xy=(2.33, 3.0), xytext=(1.67, 3.0),
            arrowprops=dict(arrowstyle="->", color="#7f8c8d", lw=2))

# arrows between RNN cells
for i in range(len(cell_xs) - 1):
    ax.annotate("", xy=(cell_xs[i+1] - 0.68, 3.0),
                xytext=(cell_xs[i] + 0.68, 3.0),
                arrowprops=dict(arrowstyle="->", color="white", lw=2))

# hidden state labels
for i in range(len(cell_xs) - 1):
    mx = (cell_xs[i] + cell_xs[i+1]) / 2
    ax.text(mx, 3.35, f"h{i+1}", ha='center', fontsize=7, color='#bdc3c7')

ax.text(6.5, 4.5, "Generated: A → al → ali → alia → ...",
        ha='center', fontsize=10, color='#2c3e50',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.7))

# legend
for text, color, y in [("Input", "#7f8c8d", 5.5),
                        ("RNN Cell", "#8e44ad", 5.0),
                        ("Output Char", "#f39c12", 4.5)]:
    draw_box(ax, 12.0, y, 1.5, 0.45, color, text, 8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "one_to_many_architecture.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved ▶  one_to_many_architecture.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "═"*65)
print("  FINAL SUMMARY")
print("═"*65)
print("\n  ── Translation Model ──")
for lbl, res in trans_results.items():
    print(f"  {lbl}: Best Acc={max(res['acc']):.4f}  Final Loss={res['loss'][-1]:.4f}")

print("\n  ── Name Generation Model ──")
for lbl, res in name_results.items():
    print(f"  {lbl}: Final Loss={res['loss'][-1]:.4f}")

print("\n  ── Sample Generated Names ──")
for _ in range(3):
    row = [generate_name(best_name_model, temperature=0.8) for _ in range(6)]
    print("  " + "  |  ".join(row))

print("\n  Output files:")
for f in ["translation_training_curves.png",
          "encoder_decoder_architecture.png",
          "namegen_analysis.png",
          "one_to_many_architecture.png"]:
    print(f"    ✔  {f}")
print("═"*65)


Device: cpu

═════════════════════════════════════════════════════════════════
  PART 1 — English → Urdu Translation (Encoder-Decoder RNN)
═════════════════════════════════════════════════════════════════
  English vocab  : 54  |  Urdu vocab : 57
  Sentence pairs : 20  |  MAX_LEN : 10
─────────────────────────────────────────────────────────────────
  Config: Small  (64u,  lr=1e-2)
─────────────────────────────────────────────────────────────────
  [Small  (64u,  lr=1e-2)] Ep   6/30 | Loss 1.0752 | Acc 0.7358
  [Small  (64u,  lr=1e-2)] Ep  12/30 | Loss 0.3262 | Acc 0.9057
  [Small  (64u,  lr=1e-2)] Ep  18/30 | Loss 0.1377 | Acc 0.9811
  [Small  (64u,  lr=1e-2)] Ep  24/30 | Loss 0.0665 | Acc 0.9906
  [Small  (64u,  lr=1e-2)] Ep  30/30 | Loss 0.0397 | Acc 0.9906

─────────────────────────────────────────────────────────────────
  Config: Medium (128u, lr=1e-3)
─────────────────────────────────────────────────────────────────
  [Medium (128u, lr=1e-3)] Ep  10/50 | Loss 2.5448 | Acc 0.3491